<div dir="rtl" align="right">
<h2>اصلاح آموزش و بررسی کامل خطاهای validation</h2>
<p>این نوت‌بوک مستقل است. GPU را فعال کن، <span dir="ltr">Run all</span> را بزن و اتصال Google Drive را تأیید کن.</p>
<p>مدل کامل قبلی با همان وزن اولیه، معماری، loss، learning rate و سه seed آموزش داده می‌شود. تغییر این آزمایش، تکمیل augmentationهای آموزشی است: چرخش و برش ملایم، تغییر روشنایی و کنتراست، نویز speckle و Gaussian ضعیف. مقادیر عددی این تبدیل‌ها انتخاب‌های ثبت‌شدهٔ این آزمایش هستند.</p>
<p>ابتدا نتایج هر سه مدل کنترل در تمام ۱۱۵ تصویر validation با گزارش قبلی تطبیق داده می‌شود. سپس یک نسخهٔ موقت برای آزمون آپدیت وزن و ذخیره/بارگذاری ساخته می‌شود. بعد، سه اجرای آموزشی جدید با seedهای 42، 43 و 44 انجام می‌شوند.</p>
<p>تصویر و ماسک با یک تبدیل هندسی تغییر می‌کنند. داده‌های validation تغییر نمی‌کنند. تصاویر test خوانده نمی‌شوند. این آزمایش تضمین نمی‌کند همهٔ خطاهای قبلی برطرف شوند؛ جدول قبل/بعد نتیجه را نشان می‌دهد.</p>
</div>


In [ ]:
#@title ۱. اتصال به Drive و بررسی GPU
from pathlib import Path
import tensorflow as tf
from google.colab import drive

if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


In [ ]:
#@title ۲. تعریف توابع مستقل آموزش
from pathlib import Path


import hashlib


import json


import math


import os


import platform


import numpy as np


import pandas as pd


from PIL import Image


import tensorflow as tf


@tf.keras.utils.register_keras_serializable(package="BUSI_MTL")
class SpatialPositionEmbedding(tf.keras.layers.Layer):
    def build(self, input_shape):
        self.positions = self.add_weight(
            name="positions", shape=(1, int(input_shape[1]), int(input_shape[2])),
            initializer=tf.keras.initializers.RandomNormal(stddev=0.02), trainable=True)
        super().build(input_shape)

    def call(self, inputs):
        return inputs + self.positions


@tf.keras.utils.register_keras_serializable(package="BUSI_MTL")
class ProbabilityToLogits(tf.keras.layers.Layer):
    def __init__(self, binary=False, epsilon=1e-6, **kwargs):
        super().__init__(**kwargs)
        self.binary, self.epsilon = binary, epsilon

    def call(self, inputs):
        p = tf.clip_by_value(inputs, self.epsilon, 1.0-self.epsilon)
        return tf.math.log(p)-tf.math.log1p(-p) if self.binary else tf.math.log(p)

    def get_config(self):
        return {**super().get_config(), "binary": self.binary, "epsilon": self.epsilon}


@tf.keras.utils.register_keras_serializable(package="BUSI_Training")
class BoundedLogWeight(tf.keras.constraints.Constraint):
    def __call__(self, value):
        return tf.clip_by_value(value, -3.0, 3.0)

    def get_config(self):
        return {}


def soft_dice_parts(y_true, y_pred):
    y_true, y_pred = tf.cast(y_true, tf.float32), tf.cast(y_pred, tf.float32)
    true_area = tf.reduce_sum(y_true, axis=(1, 2, 3))
    predicted_area = tf.reduce_sum(y_pred, axis=(1, 2, 3))
    intersection = tf.reduce_sum(y_true*y_pred, axis=(1, 2, 3))
    dice = (2*intersection+1e-6)/(true_area+predicted_area+1e-6)
    return dice, tf.cast(true_area > 0, tf.float32)


def segmentation_loss_per_image(y_true, y_pred):
    y_true, y_pred = tf.cast(y_true, tf.float32), tf.cast(y_pred, tf.float32)
    bce = tf.reduce_mean(
        tf.keras.losses.binary_crossentropy(y_true, y_pred), axis=(1, 2))
    dice, positive = soft_dice_parts(y_true, y_pred)
    return bce + positive*(1-dice)


@tf.keras.utils.register_keras_serializable(package="BUSI_Training")
class ProposedTrainer(tf.keras.Model):
    """Explicitly track task weights as trainable MODEL variables, not loss locals."""
    def __init__(self, network, class_weights, adaptive_task_weights=True, **kwargs):
        super().__init__(**kwargs)
        self.network = network
        self.class_weights = [float(x) for x in class_weights]
        self.adaptive_task_weights = bool(adaptive_task_weights)
        self.log_class = self.add_weight(
            name="log_class", shape=(), initializer="zeros", trainable=self.adaptive_task_weights,
            constraint=BoundedLogWeight())
        self.log_seg = self.add_weight(
            name="log_seg", shape=(), initializer="zeros", trainable=self.adaptive_task_weights,
            constraint=BoundedLogWeight())
        self.total_metric = tf.keras.metrics.Mean(name="loss")
        self.class_metric = tf.keras.metrics.Mean(name="classification_loss")
        self.plain_ce_metric = tf.keras.metrics.Mean(name="classification_plain_ce")
        self.seg_metric = tf.keras.metrics.Mean(name="segmentation_loss")
        self.accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy(name="classification_accuracy")
        self.dice_metric = tf.keras.metrics.Mean(name="positive_soft_dice")

    @property
    def metrics(self):
        return [self.total_metric, self.class_metric, self.plain_ce_metric,
                self.seg_metric, self.accuracy_metric, self.dice_metric]

    def build(self, input_shape):
        if not self.network.built:
            self.network.build(input_shape)
        super().build(input_shape)

    def call(self, inputs, training=False):
        return self.network(inputs, training=training)

    def _terms(self, targets, predictions):
        labels = tf.cast(tf.reshape(targets["classification"], [-1]), tf.int32)
        ce = tf.keras.losses.sparse_categorical_crossentropy(labels, predictions["classification"])
        weighted_ce = ce*tf.gather(tf.constant(self.class_weights, tf.float32), labels)
        seg = segmentation_loss_per_image(targets["segmentation"], predictions["segmentation"])
        # Chosen adaptive weighting objective; NOT a clinical uncertainty estimate.
        if self.adaptive_task_weights:
            total = (tf.exp(-self.log_class)*tf.reduce_mean(weighted_ce)+self.log_class
                     + tf.exp(-self.log_seg)*tf.reduce_mean(seg)+self.log_seg)
        else:
            total = tf.reduce_mean(weighted_ce)+tf.reduce_mean(seg)
        if self.losses:
            total += tf.add_n(self.losses)
        tf.debugging.assert_all_finite(total, "Non-finite training objective")
        return total, weighted_ce, ce, seg

    def _update(self, targets, predictions, terms):
        total, weighted_ce, ce, seg = terms
        self.total_metric.update_state(total, sample_weight=tf.cast(tf.shape(ce)[0], tf.float32))
        self.class_metric.update_state(weighted_ce)
        self.plain_ce_metric.update_state(ce)
        self.seg_metric.update_state(seg)
        self.accuracy_metric.update_state(targets["classification"], predictions["classification"])
        dice, positive = soft_dice_parts(targets["segmentation"], predictions["segmentation"])
        self.dice_metric.update_state(dice, sample_weight=positive)
        return {metric.name: metric.result() for metric in self.metrics}

    def train_step(self, data):
        images, targets, sample_weight = tf.keras.utils.unpack_x_y_sample_weight(data)
        if sample_weight is not None:
            raise ValueError("Use the recorded class weights; external sample weights are not supported.")
        with tf.GradientTape() as tape:
            predictions = self(images, training=True)
            terms = self._terms(targets, predictions)
        gradients = tape.gradient(terms[0], self.trainable_variables)
        pairs = [(gradient, variable) for gradient, variable in
                 zip(gradients, self.trainable_variables) if gradient is not None]
        self.optimizer.apply_gradients(pairs)
        return self._update(targets, predictions, terms)

    def test_step(self, data):
        images, targets, _ = tf.keras.utils.unpack_x_y_sample_weight(data)
        predictions = self(images, training=False)
        return self._update(targets, predictions, self._terms(targets, predictions))

    def get_config(self):
        return {**super().get_config(),
                "network": tf.keras.utils.serialize_keras_object(self.network),
                "class_weights": self.class_weights,
                "adaptive_task_weights": self.adaptive_task_weights}

    @classmethod
    def from_config(cls, config):
        config = dict(config)
        network = tf.keras.utils.deserialize_keras_object(config.pop("network"))
        return cls(network=network, **config)


def validation_statistics(labels, masks, probabilities):
    """Image-level validation metrics. All class orders: normal, benign, malignant."""
    labels = np.asarray(labels, dtype=np.int32)
    predicted = np.argmax(probabilities["classification"], axis=1)
    matrix = np.zeros((3, 3), dtype=np.int64)
    np.add.at(matrix, (labels, predicted), 1)
    tp = np.diag(matrix).astype(float)
    precision = np.divide(tp, matrix.sum(0), out=np.zeros(3), where=matrix.sum(0)>0)
    recall = np.divide(tp, matrix.sum(1), out=np.zeros(3), where=matrix.sum(1)>0)
    f1 = np.divide(2*precision*recall, precision+recall,
                   out=np.zeros(3), where=precision+recall>0)
    malignant = labels == 2
    normal = labels == 0
    false_malignant = int(((predicted == 2) & ~malignant).sum())
    specificity = 1-false_malignant/int((~malignant).sum())
    true_masks = np.asarray(masks)[..., 0] > 0
    predicted_masks = np.asarray(probabilities["segmentation"])[..., 0] >= 0.5
    true_area = true_masks.sum((1, 2))
    predicted_area = predicted_masks.sum((1, 2))
    intersection = (true_masks & predicted_masks).sum((1, 2))
    positive = true_area > 0
    dice = np.full(len(labels), np.nan)
    dice[positive] = 2*intersection[positive]/(true_area[positive]+predicted_area[positive])
    stats = {
        "val_accuracy": float(tp.sum()/len(labels)),
        "val_macro_f1": float(f1.mean()),
        "val_malignant_recall": float(recall[2]),
        "val_missed_malignant": int((predicted[malignant] != 2).sum()),
        "val_nonmalignant_predicted_malignant": false_malignant,
        "val_nonmalignant_specificity": float(specificity),
        "val_malignant_balanced_accuracy": float((recall[2]+specificity)/2),
        "val_normal_recall": float(recall[0]),
        "val_positive_dice": float(dice[positive].mean()),
        "val_malignant_dice": float(dice[malignant].mean()),
        "val_very_low_dice": int((dice[positive] < 0.1).sum()),
        "val_empty_positive_predictions": int((predicted_area[positive] == 0).sum()),
        "val_normal_with_any_foreground": int((predicted_area[normal] > 0).sum()),
        "val_normal_mean_foreground_percent": float(
            100*predicted_area[normal].mean()/true_masks.shape[1]/true_masks.shape[2]),
        "val_normal_max_foreground_percent": float(
            100*predicted_area[normal].max()/true_masks.shape[1]/true_masks.shape[2])}
    return stats, matrix, dice, predicted, predicted_area


def _write_json(path, value):
    path = Path(path)
    tmp = path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf8")
    os.replace(tmp, path)


def _save_model(model, path):
    path = Path(path)
    tmp = path.with_name(path.stem+".tmp.keras")
    model.save(str(tmp))
    os.replace(tmp, path)


def _selection_tuple(stats):
    # Predeclared: primary malignant-vs-other balanced accuracy; ties: Dice, macro-F1.
    return (stats["val_malignant_balanced_accuracy"],
            stats["val_positive_dice"], stats["val_macro_f1"])


class ValidationCheckpoints(tf.keras.callbacks.Callback):
    def __init__(self, directory, data, patience, initial_epoch):
        super().__init__()
        self.directory, self.data = Path(directory), data
        self.patience, self.wait = patience, 0
        self.initial_epoch = initial_epoch
        self.best = None
        self.progress = self.directory/"progress.json"
        self.best_metadata = self.directory/"best_validation.json"

    def _predict(self):
        probabilities = self.model.network.predict(self.data["X_val"], batch_size=8, verbose=0)
        for values in probabilities.values():
            if not np.isfinite(values).all():
                raise FloatingPointError("Non-finite validation predictions")
        return validation_statistics(self.data["y_val"], self.data["M_val"], probabilities)[0]

    def _save_best(self, stats, epoch_number):
        _save_model(self.model.network, self.directory/"best.keras")
        _save_model(self.model, self.directory/"best_trainer.keras")
        _write_json(self.best_metadata, {"epoch_number": epoch_number, **stats})
        self.best = _selection_tuple(stats)

    def on_train_begin(self, logs=None):
        if self.best_metadata.exists():
            self.best = _selection_tuple(json.loads(self.best_metadata.read_text()))
        else:
            initial = self._predict()
            self._save_best(initial, self.initial_epoch)
            _write_json(self.directory/"initial_validation.json", initial)
        if self.progress.exists():
            progress = json.loads(self.progress.read_text())
            if progress.get("completed_epochs") == self.initial_epoch:
                self.wait = int(progress.get("wait", 0))

    def on_epoch_end(self, epoch, logs=None):
        if logs is None:
            raise RuntimeError("Epoch logs missing")
        stats = self._predict()
        logs.update(stats)
        logs["task_weight_classification"] = float(tf.exp(-self.model.log_class).numpy())
        logs["task_weight_segmentation"] = float(tf.exp(-self.model.log_seg).numpy())
        if not np.isfinite(float(logs["loss"])):
            raise FloatingPointError("Non-finite objective")
        score = _selection_tuple(stats)
        if score > self.best:
            self._save_best(stats, epoch+1)
            self.wait = 0
        else:
            self.wait += 1
        record = {"epoch": epoch, **{k: float(v) for k, v in logs.items()}}
        _write_json(self.directory/"epochs"/f"{epoch:04d}.json", record)
        _save_model(self.model, self.directory/"last_trainer.keras")
        _write_json(self.progress, {"completed_epochs": epoch+1, "wait": self.wait})
        print("\nValidation: malignant recall {:.3f}, false malignant {}, Dice {:.3f}, task weights {:.3f}/{:.3f}".format(
            stats["val_malignant_recall"], stats["val_nonmalignant_predicted_malignant"],
            stats["val_positive_dice"], logs["task_weight_classification"], logs["task_weight_segmentation"]))
        if self.wait >= self.patience:
            self.model.stop_training = True
            print("Early stopping: no improvement under the declared selection rule.")


def _read_data(proposed_dir, manifest, root, data):
    frames = {split: manifest[manifest["split"] == split].reset_index(drop=True)
              for split in ["train", "val"]}
    labels = {"normal": 0, "benign": 1, "malignant": 2}
    source = json.loads((proposed_dir/"source_config.json").read_text())
    stats = source["normalization"]
    mean, std = np.array(stats["mean"], np.float32), np.array(stats["std"], np.float32)
    if data is None:
        print("Restoring only train/validation images and masks; no previous training is rerun.")
        data = {}
        for split, frame in frames.items():
            images = np.empty((len(frame), 224, 224, 3), np.float32)
            masks = np.empty((len(frame), 224, 224, 1), np.float32)
            for i, row in frame.iterrows():
                with Image.open(root/row["image"]) as image:
                    images[i] = np.asarray(image.convert("RGB").resize((224,224), Image.Resampling.BILINEAR))
                with Image.open(root/row["mask"]) as image:
                    masks[i,...,0] = np.asarray(image.convert("L").resize((224,224), Image.Resampling.NEAREST)) > 0
                if i % 100 == 0:
                    print(split, i+1, "/", len(frame))
            data["X_"+split] = (images-mean)/std
            data["M_"+split] = masks
            data["y_"+split] = frame["class_name"].map(labels).to_numpy(dtype=np.int32)
    else:
        data = dict(data)
        print("Using the train/validation arrays already in memory.")
    for split, frame in frames.items():
        n = len(frame)
        assert data["X_"+split].shape == (n,224,224,3)
        assert data["M_"+split].shape == (n,224,224,1)
        assert np.array_equal(data["y_"+split], frame["class_name"].map(labels).to_numpy())
        assert np.isfinite(data["X_"+split]).all()
        assert np.isin(data["M_"+split], [0,1]).all()
        assert np.array_equal(data["M_"+split].sum((1,2,3)) > 0, data["y_"+split] != 0)
    assert np.allclose(data["X_train"].mean((0,1,2), dtype=np.float64), 0, atol=0.01), "Expected training Z-score inputs."
    return data, frames


"""Train-only augmentation correction; original architecture/loss/optimizer retained.

This is an exploratory follow-up after earlier test results were inspected.
It is not an exact paper replication or a new independent test evaluation.
Only train/validation images are loaded. No labels or split assignments change.
"""
import gc
import shutil
import tempfile
import matplotlib.pyplot as plt

REPAIR_VERSION = "1.0.0"
REPAIR_NAMES = ["normal", "benign", "malignant"]
REPAIR_SEEDS = [42, 43, 44]
REPAIR_FIELDS = ["val_accuracy", "val_macro_f1", "val_malignant_recall",
    "val_nonmalignant_predicted_malignant", "val_malignant_balanced_accuracy",
    "val_positive_dice", "val_malignant_dice"]
REPAIR_AUG = {
    "horizontal_flip_probability": 0.5,
    "rotation_probability": 0.5, "rotation_degrees": [-10.0, 10.0],
    "crop_probability": 0.5, "crop_side_fraction": [0.95, 1.0],
    "brightness_probability": 0.5, "brightness_offset_255": [-12.75, 12.75],
    "contrast_probability": 0.5, "contrast_factor": [0.9, 1.1],
    "speckle_probability": 0.5, "speckle_sigma": [0.0, 0.05],
    "gaussian_probability": 0.5, "gaussian_sigma_255": [0.0, 2.55],
    "image_interpolation": "PIL BILINEAR", "mask_interpolation": "PIL NEAREST",
    "rotation_fill": 0, "noise_channels": "one shared noise field across RGB",
    "geometry_guard": "revert geometry if it removes every lesion pixel",
    "scope": "training only; independently resampled each epoch",
    "parameter_origin": "augmentation types from proposal; numerical ranges are declared experimental choices",
}


def _repair_sha(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024*1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _repair_guard(path, value):
    if path.is_file():
        if json.loads(path.read_text()) != value:
            raise ValueError("این پوشه مربوط به تنظیمات دیگری است؛ خروجی قبلی را حفظ کن: "+str(path))
    else:
        _write_json(path, value)


def _repair_augment(raw, mask, seed, epoch, index, spec=None):
    """Stateless paired geometry; grayscale-preserving image-only photometrics."""
    spec = REPAIR_AUG if spec is None else spec
    raw = np.asarray(raw, np.uint8)
    mask = (np.asarray(mask)[...,0] > 0).astype(np.uint8)
    if raw.shape != (224,224,3) or mask.shape != (224,224):
        raise ValueError("Expected 224x224 RGB image and single-channel mask")
    rng = np.random.default_rng(np.random.SeedSequence([int(seed),int(epoch),int(index),9183]))
    image, target = Image.fromarray(raw), Image.fromarray(mask)
    if rng.random() < spec["horizontal_flip_probability"]:
        image, target = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT), target.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
    if rng.random() < spec["rotation_probability"]:
        angle = float(rng.uniform(*spec["rotation_degrees"]))
        image = image.rotate(angle, resample=Image.Resampling.BILINEAR, expand=False, fillcolor=(0,0,0))
        target = target.rotate(angle, resample=Image.Resampling.NEAREST, expand=False, fillcolor=0)
    if rng.random() < spec["crop_probability"]:
        side = int(round(224*float(rng.uniform(*spec["crop_side_fraction"]))))
        side = min(224,max(1,side))
        left, top = [int(rng.integers(0,225-side)) for _ in range(2)]
        box = (left,top,left+side,top+side)
        image = image.crop(box).resize((224,224),Image.Resampling.BILINEAR)
        target = target.crop(box).resize((224,224),Image.Resampling.NEAREST)
    target = np.asarray(target,dtype=np.float32)[...,None]
    image = np.asarray(image,dtype=np.float32)
    if mask.any() and not target.any():
        image, target = raw.astype(np.float32), mask.astype(np.float32)[...,None]
    if rng.random() < spec["contrast_probability"]:
        factor = float(rng.uniform(*spec["contrast_factor"]))
        center = image.mean((0,1),keepdims=True,dtype=np.float32)
        image = (image-center)*np.float32(factor)+center
    if rng.random() < spec["brightness_probability"]:
        image = image+np.float32(rng.uniform(*spec["brightness_offset_255"]))
    if rng.random() < spec["speckle_probability"]:
        sigma = np.float32(rng.uniform(*spec["speckle_sigma"]))
        image = image*(1+sigma*rng.standard_normal((224,224,1),dtype=np.float32))
    if rng.random() < spec["gaussian_probability"]:
        sigma = np.float32(rng.uniform(*spec["gaussian_sigma_255"]))
        image = image+sigma*rng.standard_normal((224,224,1),dtype=np.float32)
    image = np.clip(image,0,255).astype(np.float32)
    if not np.isfinite(image).all() or not np.isin(target,[0,1]).all():
        raise FloatingPointError("Invalid augmentation result")
    if bool(mask.any()) != bool(target.any()):
        raise ValueError("Augmentation changed lesion presence")
    return image,target


def _repair_epoch_arrays(data, mean, std, seed, epoch):
    n = len(data["y_train"])
    order = np.random.default_rng(np.random.SeedSequence([seed,epoch,734])).permutation(n)
    images, masks = np.empty_like(data["X_train"]), np.empty_like(data["M_train"])
    for position,index in enumerate(order):
        raw,mask = _repair_augment(data["raw_train"][index],data["M_train"][index],seed,epoch,int(index))
        images[position],masks[position] = (raw-mean)/std, mask
    return images,masks,data["y_train"][order]


def _repair_prediction(network, data):
    values = network.predict(data["X_val"],batch_size=8,verbose=0)
    if not isinstance(values,dict) or set(values) != {"classification","segmentation"}:
        raise ValueError("Expected named classification/segmentation outputs")
    p,mask = [np.asarray(values[key]) for key in ["classification","segmentation"]]
    if p.shape != (115,3) or mask.shape != (115,224,224,1):
        raise ValueError("Unexpected validation output shapes")
    if not np.isfinite(p).all() or not np.isfinite(mask).all() or (p<0).any() or (p>1).any() or (mask<0).any() or (mask>1).any():
        raise FloatingPointError("Invalid validation probabilities")
    np.testing.assert_allclose(p.sum(1),1,atol=1e-5,rtol=0)
    return values


def _repair_per_image(frame, data, values, size_cuts):
    stats,matrix,dice,predicted,predicted_area = validation_statistics(data["y_val"],data["M_val"],values)
    truth = data["y_val"]
    p = np.asarray(values["classification"])
    area = data["M_val"].sum((1,2,3))
    result = frame[["image","mask","class_name","group_id"]].copy()
    result["predicted_class"] = [REPAIR_NAMES[i] for i in predicted]
    result["classification_error"] = predicted != truth
    result["confidence"] = p.max(1)
    result["probability_of_true_class"] = p[np.arange(len(truth)),truth]
    for i,name in enumerate(REPAIR_NAMES):
        result["p_"+name] = p[:,i]
    result["reference_pixels"] = area.astype(int)
    result["reference_area_percent"] = area*100/(224*224)
    result["predicted_pixels"] = predicted_area
    result["dice"] = dice
    result["normal_false_foreground_percent"] = np.where(truth==0,predicted_area*100/(224*224),np.nan)
    result["normal_class_with_mask"] = (predicted==0) & (predicted_area>0)
    result["lesion_class_without_mask"] = (predicted!=0) & (predicted_area==0)
    stats["val_normal_class_with_mask"] = int(result["normal_class_with_mask"].sum())
    stats["val_lesion_class_without_mask"] = int(result["lesion_class_without_mask"].sum())
    stats["val_inconsistent_pairs"] = stats["val_normal_class_with_mask"]+stats["val_lesion_class_without_mask"]
    bins = np.searchsorted(size_cuts,area/(224*224),side="right")
    result["lesion_size_bin"] = ["no_lesion" if a==0 else "Q"+str(int(b)+1) for a,b in zip(area,bins)]
    return stats,matrix,result


def _repair_save_predictions(folder, frame, data, values, size_cuts, condition, seed, epoch):
    folder.mkdir(parents=True,exist_ok=True)
    stats,matrix,per_image = _repair_per_image(frame,data,values,size_cuts)
    per_image.to_csv(folder/"validation_per_image.csv",index=False)
    pd.DataFrame(matrix,index=REPAIR_NAMES,columns=REPAIR_NAMES).to_csv(folder/"validation_confusion.csv")
    binary = np.asarray(values["segmentation"])[...,0] >= 0.5
    temporary = folder/"validation_predictions.tmp.npz"
    np.savez_compressed(temporary,probabilities=np.asarray(values["classification"],np.float32),
        mask_bits=np.packbits(binary.reshape(len(binary),-1),axis=1),
        images=frame["image"].to_numpy(dtype=str),mask_shape=np.array([224,224],np.int32))
    os.replace(temporary,folder/"validation_predictions.npz")
    row = {"condition":condition,"seed":seed,"selected_epoch":int(epoch),**stats}
    _write_json(folder/"summary.json",row)
    return row,per_image


def _repair_check_baseline_config(config, initial_sha, weights):
    expected = {"source_sha256":initial_sha,"class_order":REPAIR_NAMES,
        "class_cost_ratio":[1,1,2],"cnn_encoder_trainable":False,
        "optimizer":"Adam","learning_rate":1e-4,"clipnorm":1.0,
        "batch_size":8,"max_epochs":30,"patience":8,
        "augmentation":"paired_horizontal_flip_probability_0.5",
        "selection_priority":["malignant_vs_others_balanced_accuracy","positive_hard_dice","macro_f1"],
        "adaptive_loss":"exp(-s_cls)*weighted_CE+s_cls+exp(-s_seg)*seg_loss+s_seg",
        "adaptive_log_weight_bounds":[-3,3],"initial_task_weights":[1,1],
        "segmentation_loss":"BCE_all_plus_Dice_on_positive_images"}
    different = {k:(config.get(k),v) for k,v in expected.items() if config.get(k) != v}
    if different:
        raise ValueError("تنظیمات کنترل با این آزمایش همسان نیست: "+str(different))
    np.testing.assert_allclose(config["normalized_class_weights"],weights,atol=1e-6,rtol=0)


def _repair_choose_protocol(experiments_root, evaluation_dir):
    if evaluation_dir is not None:
        return Path(evaluation_dir)
    found = sorted(Path(experiments_root).glob("proposed_model_*/ablations/component_study_v1/final_evaluation_v1/frozen_protocol.json"))
    if not found:
        raise FileNotFoundError("پروژهٔ ارزیابی نهایی قبلی پیدا نشد: "+str(experiments_root))
    for i,path in enumerate(found):
        print(f"[{i}] {path.parent}")
    answer = "0" if len(found)==1 else input("شمارهٔ پروژهٔ قبلی را وارد کن: ").strip()
    if not answer.isdigit() or not 0<=int(answer)<len(found):
        raise ValueError("شمارهٔ پروژه معتبر نیست")
    return found[int(answer)].parent


def _repair_preview(data, mean, std, path):
    indices = [int(np.flatnonzero(data["y_train"]==i)[0]) for i in range(3)]
    fig,axes = plt.subplots(3,2,figsize=(6,8))
    for row,index in enumerate(indices):
        augmented,mask = _repair_augment(data["raw_train"][index],data["M_train"][index],42,0,index)
        for column,(raw,target) in enumerate([(data["raw_train"][index],data["M_train"][index]),(augmented,mask)]):
            axes[row,column].imshow(np.clip(raw/255,0,1))
            target = target[...,0] > 0
            if target.any():
                axes[row,column].imshow(np.ma.masked_where(~target,target),cmap="Reds",vmin=0,vmax=1,alpha=.4)
            axes[row,column].set_title(REPAIR_NAMES[row]+(" / original" if column==0 else " / augmented"))
            axes[row,column].axis("off")
    fig.tight_layout()
    fig.savefig(path,dpi=160,bbox_inches="tight",facecolor="white")
    plt.close(fig)


def _repair_smoke(initial_path, data, mean, std, weights, report):
    """Real Colab forward/backward and optimizer save/reload before long runs."""
    tf.keras.utils.set_random_seed(42)
    network = tf.keras.models.load_model(str(initial_path),compile=False)
    network.get_layer("shared_encoder").trainable = False
    trainer = ProposedTrainer(network,weights.tolist(),name="augmentation_smoke")
    trainer(data["X_train"][:2],training=False)
    trainer.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4,clipnorm=1.0))
    indices = [int(np.flatnonzero(data["y_train"]==i)[0]) for i in range(3)]
    batch = [_repair_augment(data["raw_train"][i],data["M_train"][i],42,0,i) for i in indices]
    x = np.stack([(raw-mean)/std for raw,_ in batch])
    target = {"classification":data["y_train"][indices],"segmentation":np.stack([mask for _,mask in batch])}
    encoder = network.get_layer("shared_encoder")
    before = [w.numpy().copy() for w in encoder.weights]
    trainable_before = [w.numpy().copy() for w in network.trainable_weights]
    logs = trainer.train_on_batch(x,target,return_dict=True)
    if not all(np.isfinite(float(v)) for v in logs.values()):
        raise FloatingPointError("Non-finite smoke-test training metrics")
    if not any(not np.array_equal(a,w.numpy()) for a,w in zip(trainable_before,network.trainable_weights)):
        raise RuntimeError("No trainable network weights changed in the smoke test")
    if any(not np.array_equal(a,w.numpy()) for a,w in zip(before,encoder.weights)):
        raise RuntimeError("Frozen encoder weights or BatchNorm state changed")
    expected = trainer(x,training=False)
    with tempfile.TemporaryDirectory(prefix="busi_repair_smoke_") as directory:
        file = Path(directory)/"trainer.keras"
        _save_model(trainer,file)
        restored = tf.keras.models.load_model(str(file))
        observed = restored(x,training=False)
        for key in expected:
            np.testing.assert_allclose(expected[key].numpy(),observed[key].numpy(),atol=1e-6,rtol=1e-6)
        if int(restored.optimizer.iterations.numpy()) != 1:
            raise RuntimeError("Optimizer state did not survive save/reload")
        del restored
    _write_json(report/"colab_smoke_test.json",{"forward_backward":"passed","encoder_unchanged":True,
        "trainable_network_weights_changed":True,"trainer_save_reload":"passed","optimizer_iteration":1,
        "logs":{k:float(v) for k,v in logs.items()}})
    del trainer,network,expected,observed,before,trainable_before
    gc.collect()
    print("Colab smoke test passed: paired inputs, backward update, frozen encoder, optimizer save/reload.",flush=True)


def _repair_train_seed(out, report, source, initial_path, data, frame, mean, std, size_cuts, weights, seed):
    directory = out/f"training_seed{seed}"
    directory.mkdir(exist_ok=True)
    (directory/"epochs").mkdir(exist_ok=True)
    _repair_guard(directory/"config.json",{"experiment_settings_sha256":_repair_sha(out/"settings.json"),
        "seed":seed,"initial_sha256":_repair_sha(initial_path)})
    destination = report/f"augmented_seed{seed}"
    if (directory/"completed.json").is_file():
        completed = json.loads((directory/"completed.json").read_text())
        if _repair_sha(directory/"best.keras") != completed["model_sha256"]:
            raise ValueError("Saved augmented model changed: "+str(directory))
        return json.loads((destination/"summary.json").read_text()),pd.read_csv(destination/"validation_per_image.csv")
    tf.keras.utils.set_random_seed(seed)
    steps = math.ceil(len(data["y_train"])/8)
    last = directory/"last_trainer.keras"
    if last.is_file():
        trainer = tf.keras.models.load_model(str(last))
        if trainer.network.get_layer("shared_encoder").trainable:
            raise ValueError("The resumed encoder is unexpectedly trainable")
        iterations = int(trainer.optimizer.iterations.numpy())
        if iterations % steps:
            raise ValueError("Last checkpoint is not at an epoch boundary")
        start = iterations//steps
        for epoch in range(start):
            if not (directory/"epochs"/f"{epoch:04d}.json").is_file():
                raise FileNotFoundError("Missing completed epoch record: "+str(epoch))
        print("Resuming seed",seed,"after epoch",start,flush=True)
    else:
        network = tf.keras.models.load_model(str(initial_path),compile=False)
        network.get_layer("shared_encoder").trainable = False
        trainer = ProposedTrainer(network,weights.tolist(),name="augmented_trainer")
        trainer(data["X_train"][:1],training=False)
        trainer.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4,clipnorm=1.0))
        del network
        start = 0
    val = tf.data.Dataset.from_tensor_slices((data["X_val"],{
        "classification":data["y_val"],"segmentation":data["M_val"]})).batch(8)
    monitor = ValidationCheckpoints(directory,data,patience=8,initial_epoch=start)
    for epoch in range(start,30):
        progress = json.loads((directory/"progress.json").read_text()) if (directory/"progress.json").is_file() else {}
        if progress.get("completed_epochs")==epoch and progress.get("wait",0)>=8:
            break
        print(f"Seed {seed}: preparing paired augmentation for epoch {epoch+1}/30",flush=True)
        x,m,y = _repair_epoch_arrays(data,mean,std,seed,epoch)
        train = tf.data.Dataset.from_tensor_slices((x,{"classification":y,"segmentation":m})).batch(8).prefetch(1)
        monitor.initial_epoch = epoch
        trainer.fit(train,validation_data=val,initial_epoch=epoch,epochs=epoch+1,
            callbacks=[monitor],verbose=2)
        if int(trainer.optimizer.iterations.numpy()) != (epoch+1)*steps:
            raise RuntimeError("Training step count differs from the fixed epoch size")
        records = [json.loads((directory/"epochs"/f"{e:04d}.json").read_text()) for e in range(epoch+1)]
        pd.DataFrame(records).to_csv(directory/"training_log.csv",index=False)
        del train,x,m,y
        gc.collect()
        if trainer.stop_training:
            break
    del trainer,val,monitor
    gc.collect()
    best = tf.keras.models.load_model(str(directory/"best.keras"),compile=False)
    values = _repair_prediction(best,data)
    metadata = json.loads((directory/"best_validation.json").read_text())
    stats = validation_statistics(data["y_val"],data["M_val"],values)[0]
    for field in REPAIR_FIELDS:
        np.testing.assert_allclose(stats[field],metadata[field],atol=1e-5,rtol=0)
    row,per_image = _repair_save_predictions(destination,frame,data,values,size_cuts,
        "proposal_augmentation",seed,int(metadata["epoch_number"]))
    _write_json(directory/"completed.json",{"selected_epoch":int(metadata["epoch_number"]),
        "model_sha256":_repair_sha(directory/"best.keras"),"test_evaluated":False})
    for filename in ["best_validation.json","initial_validation.json","training_log.csv"]:
        if (directory/filename).is_file():
            shutil.copyfile(directory/filename,destination/filename)
    if row["selected_epoch"] == 0:
        print("Initial network retained for seed",seed,"; do not describe this seed as successful training.",flush=True)
    del best,values
    gc.collect()
    return row,per_image


def _repair_tables(report, rows, per_images):
    summary = pd.DataFrame(rows)
    summary.to_csv(report/"validation_comparison.csv",index=False)
    metrics = ["val_accuracy","val_macro_f1","val_malignant_recall","val_nonmalignant_specificity",
        "val_nonmalignant_predicted_malignant","val_malignant_balanced_accuracy","val_positive_dice",
        "val_malignant_dice","val_normal_mean_foreground_percent","val_inconsistent_pairs"]
    mean_std = summary.groupby("condition",sort=False)[metrics].agg(["mean","std"])
    mean_std.to_csv(report/"validation_mean_std.csv")
    before = summary[summary["condition"]=="original_horizontal_flip"].set_index("seed")
    after = summary[summary["condition"]=="proposal_augmentation"].set_index("seed")
    delta = after[metrics]-before[metrics]
    delta.to_csv(report/"paired_validation_changes.csv")
    all_images = pd.concat(per_images,ignore_index=True)
    all_images.to_csv(report/"validation_all_images.csv",index=False)
    all_images.loc[all_images["classification_error"]].sort_values("confidence",ascending=False).to_csv(
        report/"classification_errors.csv",index=False)
    grouped = all_images.groupby(["condition","seed","class_name"],sort=False)
    by_class = grouped.agg(images=("image","size"),classification_errors=("classification_error","sum"),
        mean_dice=("dice","mean"),normal_mean_false_foreground_percent=("normal_false_foreground_percent","mean"))
    by_class.to_csv(report/"errors_by_class.csv")
    all_images[all_images["class_name"]!="normal"].groupby(["condition","seed","lesion_size_bin"],sort=False).agg(
        images=("image","size"),classification_errors=("classification_error","sum"),mean_dice=("dice","mean")
    ).to_csv(report/"errors_by_lesion_size.csv")
    return summary,mean_std


def _repair_comparison_figures(report, frame, data, mean, std):
    before_folder,after_folder = report/"control_seed42",report/"augmented_seed42"
    per_image = pd.read_csv(before_folder/"validation_per_image.csv")
    # Examples are selected from the control BEFORE inspecting the new model.
    errors = per_image[per_image["classification_error"]].sort_values("confidence",ascending=False).head(6).index.tolist()
    normals = per_image[per_image["class_name"]=="normal"].sort_values("predicted_pixels",ascending=False).head(3).index.tolist()
    indices = list(dict.fromkeys(errors+normals))
    per_image.iloc[indices][["image","class_name","group_id"]].to_csv(report/"diagnostic_cases.csv",index=False)
    arrays = []
    for folder in [before_folder,after_folder]:
        with np.load(folder/"validation_predictions.npz",allow_pickle=False) as npz:
            p = npz["probabilities"]
            masks = np.unpackbits(npz["mask_bits"],axis=1,count=224*224).reshape(-1,224,224).astype(bool)
            np.testing.assert_array_equal(npz["images"],frame["image"].to_numpy())
            arrays.append((p,masks))
    folder = report/"figures"
    folder.mkdir(exist_ok=True)
    for number,index in enumerate(indices,1):
        raw = np.clip((data["X_val"][index]*std+mean)/255,0,1)
        fig,axes = plt.subplots(1,3,figsize=(10,3.6))
        axes[0].imshow(raw)
        truth = data["M_val"][index,...,0]>0
        if truth.any():
            axes[0].imshow(np.ma.masked_where(~truth,truth),cmap="Reds",vmin=0,vmax=1,alpha=.45)
        axes[0].set_title("Reference: "+frame.iloc[index]["class_name"])
        for column,(label,(p,masks)) in enumerate(zip(["Original","Augmented"],arrays),1):
            axes[column].imshow(raw)
            if masks[index].any():
                axes[column].imshow(np.ma.masked_where(~masks[index],masks[index]),cmap="Blues",vmin=0,vmax=1,alpha=.45)
            axes[column].set_title(label+": "+REPAIR_NAMES[int(p[index].argmax())]+f"\np={p[index].max():.3f}")
        for ax in axes:
            ax.axis("off")
        fig.suptitle(frame.iloc[index]["image"],fontsize=10)
        fig.tight_layout(rect=(0,0,1,.92))
        fig.savefig(folder/f"case_{number:02d}.png",dpi=160,bbox_inches="tight",facecolor="white")
        plt.close(fig)


def run_busi_validation_repair(evaluation_dir=None,
        dataset_root="/content/drive/MyDrive/BUSI_MERGED",
        experiments_root="/content/drive/MyDrive/BUSI_EXPERIMENTS"):
    """One declared three-seed augmentation correction; validation-only comparison."""
    if not tf.config.list_physical_devices("GPU"):
        raise RuntimeError("پیش از آموزش، GPU را در تنظیمات Colab فعال کن.")
    evaluation_dir = _repair_choose_protocol(experiments_root,evaluation_dir)
    root = Path(dataset_root)
    protocol_file = evaluation_dir/"frozen_protocol.json"
    protocol = json.loads(protocol_file.read_text())
    if _repair_sha(protocol_file) != json.loads((evaluation_dir/"completed.json").read_text())["protocol_sha256"]:
        raise ValueError("Frozen evaluation protocol changed")
    if protocol["class_order"] != REPAIR_NAMES or protocol["mask_threshold"] != .5:
        raise ValueError("Unexpected class order or mask threshold")
    baseline = [next(c for c in protocol["checkpoints"] if c["variant"]=="full_lprm_v1" and c["seed"]==seed) for seed in REPAIR_SEEDS]
    sources = {str(Path(c["model"]).parent.parent) for c in baseline}
    if len(sources) != 1:
        raise ValueError("The control models do not share one source project")
    source = Path(sources.pop())
    initial_path = source/"initial.keras"
    initial_sha = _repair_sha(initial_path)
    manifest_path = source/"split_manifest.csv"
    if _repair_sha(manifest_path) != protocol["manifest_sha256"]:
        raise ValueError("Fixed split changed")
    manifest = pd.read_csv(manifest_path)
    if len(manifest)!=761 or not manifest["image"].is_unique or manifest["split"].value_counts().to_dict()!={"train":499,"val":115,"test":147}:
        raise ValueError("Unexpected fixed image split")
    if manifest["group_id"].isna().any() or manifest.groupby("group_id")["split"].nunique().max()!=1:
        raise ValueError("Similarity groups cross split boundaries")
    for column in ["image","mask"]:
        if any(Path(p).is_absolute() or ".." in Path(p).parts for p in manifest[column]):
            raise ValueError("Expected relative image/mask paths")
    normalization = json.loads((source/"source_config.json").read_text())["normalization"]
    for key in ["mean","std"]:
        np.testing.assert_allclose(normalization[key],protocol["normalization"][key],atol=1e-7,rtol=0)
    mean,std = [np.asarray(normalization[key],np.float32) for key in ["mean","std"]]
    if mean.shape!=(3,) or std.shape!=(3,) or not np.isfinite(mean).all() or not np.isfinite(std).all() or (std<=0).any():
        raise ValueError("Invalid saved training normalization")
    train_labels = manifest[manifest["split"]=="train"]["class_name"].map(dict(zip(REPAIR_NAMES,range(3)))).to_numpy(np.int32)
    raw_weights = np.array([1,1,2],np.float32)
    weights = raw_weights/raw_weights[train_labels].mean()
    controls = []
    for item in baseline:
        if _repair_sha(item["model"]) != item["model_sha256"]:
            raise ValueError("Original control model changed")
        config_file = Path(item["model"]).parent/"config.json"
        config = json.loads(config_file.read_text())
        _repair_check_baseline_config(config,initial_sha,weights)
        if int(config["seed"]) != item["seed"]:
            raise ValueError("Control seed metadata differs")
        controls.append({"seed":item["seed"],"model_sha256":item["model_sha256"],"config_sha256":_repair_sha(config_file)})
    out = evaluation_dir.parent/"augmentation_repair_v1"
    out.mkdir(exist_ok=True)
    report = out/"reports"
    report.mkdir(exist_ok=True)
    settings = {"version":REPAIR_VERSION,"purpose":"train-only augmentation correction; no claim of guaranteed error removal",
        "source_initial":str(initial_path),"source_initial_sha256":initial_sha,
        "frozen_protocol_sha256":_repair_sha(protocol_file),"split_sha256":_repair_sha(manifest_path),
        "source_config_sha256":_repair_sha(source/"source_config.json"),"controls":controls,
        "class_order":REPAIR_NAMES,"seeds":REPAIR_SEEDS,"normalization":normalization,
        "augmentation":REPAIR_AUG,"architecture":"unchanged full_lprm_v1; shared_encoder frozen",
        "optimizer":"Adam","learning_rate":1e-4,"clipnorm":1.,"batch_size":8,"epochs":30,"patience":8,
        "class_cost_ratio":[1,1,2],"loss":"unchanged BCE+positive Dice / weighted CE / learned task weights",
        "selection":["malignant_vs_other_balanced_accuracy","positive_Dice","macro_F1"],
        "comparison":"same saved initial network and seed-specific original full-model controls; warm-start continuation seeds",
        "test_history":"earlier test results have already been inspected; this is an exploratory follow-up, not a new independent test",
        "test_images_loaded":False,"resume":"optimizer and epoch state resume; dropout randomness is not restored bit-for-bit",
        "versions":{"tensorflow":tf.__version__,"keras":tf.keras.__version__,"numpy":np.__version__,"python":platform.python_version()}}
    _repair_guard(out/"settings.json",settings)
    shutil.copyfile(out/"settings.json",report/"settings.json")
    if (out/"completed.json").is_file():
        complete = json.loads((out/"completed.json").read_text())
        for relative in complete["report_files"]:
            if not (report/relative).is_file():
                raise FileNotFoundError("Missing completed report: "+relative)
        print("This correction experiment is already complete; no training rerun.")
        archive = shutil.make_archive("/content/BUSI_Validation_Repair_RESULTS","zip",root_dir=report)
        return {"directory":out,"comparison":pd.read_csv(report/"validation_comparison.csv"),"zip_path":archive}
    data,frames = _read_data(source,manifest,root,None)
    for split,counts in [("train",{"normal":91,"benign":281,"malignant":127}),("val",{"normal":16,"benign":64,"malignant":35})]:
        if frames[split]["class_name"].value_counts().to_dict()!=counts:
            raise ValueError("Unexpected class counts in "+split)
    data["raw_train"] = np.rint(data["X_train"]*std+mean).clip(0,255).astype(np.uint8)
    train_areas = data["M_train"].sum((1,2,3))/(224*224)
    size_cuts = np.quantile(train_areas[train_areas>0],[.25,.5,.75])
    _write_json(report/"lesion_size_bins.json",{"source":"positive training masks only","quartile_cutpoints_fraction":size_cuts.tolist()})
    np.savez_compressed(report/"validation_reference.npz",images=frames["val"]["image"].to_numpy(dtype=str),
        labels=data["y_val"],mask_bits=np.packbits((data["M_val"][...,0]>0).reshape(115,-1),axis=1),
        mask_shape=np.array([224,224],np.int32))
    _repair_preview(data,mean,std,report/"augmentation_preview.png")
    rows,images,checks = [],[],[]
    # Audit every validation image under all three frozen full-model controls.
    for item in baseline:
        print("Checking original validation model, seed",item["seed"],flush=True)
        network = tf.keras.models.load_model(item["model"],compile=False)
        values = _repair_prediction(network,data)
        stats = validation_statistics(data["y_val"],data["M_val"],values)[0]
        for field in REPAIR_FIELDS:
            expected = item["validation"][field]
            checks.append({"seed":item["seed"],"metric":field,"expected":expected,"observed":stats[field],
                "matches":bool(np.isclose(expected,stats[field],atol=1e-5,rtol=0))})
        pd.DataFrame(checks).to_csv(report/"original_validation_recheck.csv",index=False)
        if not all(row["matches"] for row in checks):
            raise ValueError("Original validation predictions differ; training has not started")
        row,per_image = _repair_save_predictions(report/f"control_seed{item['seed']}",frames["val"],data,values,
            size_cuts,"original_horizontal_flip",item["seed"],item["selected_epoch"])
        rows.append(row)
        per_image.insert(0,"condition",row["condition"]);per_image.insert(1,"seed",item["seed"])
        images.append(per_image)
        del network,values
        gc.collect()
    # One real gradient/serialization check uses a disposable network.
    if not (report/"colab_smoke_test.json").is_file():
        _repair_smoke(initial_path,data,mean,std,weights,report)
    for seed in REPAIR_SEEDS:
        print("Training the augmentation correction, seed",seed,flush=True)
        row,per_image = _repair_train_seed(out,report,source,initial_path,data,frames["val"],mean,std,size_cuts,weights,seed)
        rows.append(row)
        per_image.insert(0,"condition",row["condition"]);per_image.insert(1,"seed",seed)
        images.append(per_image)
        pd.DataFrame(rows).to_csv(report/"validation_comparison_progress.csv",index=False)
    summary,mean_std = _repair_tables(report,rows,images)
    _repair_comparison_figures(report,frames["val"],data,mean,std)
    for item in baseline:
        if _repair_sha(item["model"]) != item["model_sha256"]:
            raise RuntimeError("An original control checkpoint changed during the run")
    if _repair_sha(initial_path) != initial_sha:
        raise RuntimeError("The original initial model changed")
    notes = ("Augmentation correction: same full_lprm_v1 architecture, initial weights, loss and optimizer as controls.\n"
        "Original controls: horizontal flip only. New training: declared paired geometric and image-only photometric augmentation.\n"
        "Numerical augmentation ranges are experimental choices, not values claimed from the base paper.\n"
        "Seeds 42,43,44 share a warm-start network; they vary continuation randomness.\n"
        "All checkpoint selection uses validation only. Earlier test results were already seen; this follow-up is exploratory.\n"
        "No images are removed or relabeled. No inference-time class gating or mask cleanup is applied.\n"
        "All validation examples are reported; selected illustrations are chosen from original control errors only.\n"
        "A retained initial model or a worse metric must be reported as such. Improved performance is not guaranteed.\n"
        "The downloadable report excludes model and optimizer checkpoints, which stay in the Drive experiment folder.\n"
        "Implementation references: https://keras.io/guides/custom_train_step_in_tensorflow/ ; https://pillow.readthedocs.io/en/stable/reference/Image.html\n")
    (report/"README.txt").write_text(notes,encoding="utf-8")
    _write_json(out/"completed.json",{"version":REPAIR_VERSION,"test_evaluated":False,"seeds":REPAIR_SEEDS,
        "report_files":[str(p.relative_to(report)) for p in sorted(report.rglob("*")) if p.is_file()]})
    archive = shutil.make_archive("/content/BUSI_Validation_Repair_RESULTS","zip",root_dir=report)
    print(summary[["condition","seed","selected_epoch","val_accuracy","val_malignant_recall","val_positive_dice","val_normal_mean_foreground_percent"]].to_string(index=False))
    print("Saved:",out)
    print("No test images loaded. Send the downloaded results ZIP for the next decision.")
    return {"directory":out,"comparison":summary,"mean_std":mean_std,"zip_path":archive}


In [ ]:
#@title ۳. کنترل، آموزش اصلاح‌شده و دریافت نتیجه
from IPython.display import display
from google.colab import files

repair_result = run_busi_validation_repair()

columns = ["condition", "seed", "selected_epoch", "val_accuracy",
           "val_malignant_recall", "val_nonmalignant_predicted_malignant",
           "val_positive_dice", "val_normal_mean_foreground_percent"]
display(repair_result["comparison"][columns])
print("Saved experiment:", repair_result["directory"])
files.download(repair_result["zip_path"])


<div dir="rtl" align="right">
<p>حداکثر ۳۰ epoch برای هر seed اجرا می‌شود؛ توقف زودهنگام طبق همان معیار قبلی فعال است. اگر اتصال قطع شد، همین نوت‌بوک را دوباره اجرا کن تا از checkpoint آخر ادامه دهد.</p>
<p>مدل‌ها و نتایج این آزمایش در پوشهٔ <code dir="ltr">augmentation_repair_v1</code> ذخیره می‌شوند. نتایج قبلی حفظ می‌شوند. فایل دانلودی <code dir="ltr">BUSI_Validation_Repair_RESULTS.zip</code> شامل جدول مقایسه، تمام خطاهای validation، ماسک‌ها و شکل‌های قبل/بعد است؛ آن را برای بررسی بعدی بفرست.</p>
<p>این مقایسه ادامهٔ اکتشافی کار پس از مشاهدهٔ نتایج test قبلی است و آزمون مستقل جدید محسوب نمی‌شود. ساختار و بخش‌های NumPy/Pillow این کد بررسی شده‌اند؛ آزمون واقعی TensorFlow در سلول سوم و پیش از آموزش‌های طولانی اجرا می‌شود.</p>
</div>
